# get-children-callable-param — worked example 3: __call__ delegates to forward

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `get-children-callable-param`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The `nn.Module` callable convention: the base class defines `__call__` to delegate to `self.forward(*args, **kwargs)`, and `forward` itself raises `NotImplementedError` so every subclass is forced to define its own compute. This is why you call `model(x)` rather than `model.forward(x)`.

## Worked solution

We build the base `Module` with the call-to-forward delegation.

1. `__call__(self, *args, **kwargs)` forwards every positional and keyword argument verbatim to `self.forward(*args, **kwargs)` and returns its result. This indirection is where `nn.Module` would also run hooks.
2. `forward(self, *args, **kwargs)` on the base raises `NotImplementedError`, with a message that interpolates `type(self).__name__`. Using the dynamic type name means the error correctly names the *subclass* that forgot to implement forward.
3. A subclass that defines `forward` becomes callable: `instance(x)` returns whatever its `forward` returns.
4. A subclass that does NOT override `forward` raises the informative error when called. We demonstrate both paths.

In [ ]:
class Module:
    def __call__(self, *args, **kwargs):
        return self.forward(*args, **kwargs)

    def forward(self, *args, **kwargs):
        raise NotImplementedError(f'{type(self).__name__} must implement forward')

class Doubler(Module):
    def forward(self, x):
        return x * 2

class Forgot(Module):
    pass

print('doubler:', Doubler()(21))
try:
    Forgot()(1)
except NotImplementedError as e:
    print('raised:', str(e))